<a href="https://colab.research.google.com/github/ayushdwivedi3305-commits/My-everything-/blob/main/2_03_practice_bike_hire_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [2]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [3]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [4]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*

1. bike_type has 7 spellings for 2 types (case, extra space, "E-bike") → standardise to Electric / Standard
2. 12 duplicated trips (612 rows, 600 IDs) → remove exact duplicates
3. 30 trips have no duration → drop them (can't average a blank)
4. Max duration 2,880 min (48 h) → impossible ride, remove trips ≥ 24 h
5. start_time in two formats → parse both formats explicitly
6. 1 station missing capacity → ignore, not needed for this question

In [8]:
for name, df in tables.items():
    print(name, df.shape, df.isna().sum().to_dict())
display(q("SELECT bike_type, COUNT(*) AS n FROM bikes GROUP BY bike_type ORDER BY n DESC"))
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_trips FROM trips"))
display(q("SELECT MIN(duration_min), MEDIAN(duration_min), MAX(duration_min) FROM trips"))
display(q("SELECT start_time FROM trips LIMIT 3"))
display(q("SELECT start_time FROM trips WHERE start_time LIKE '%/%' LIMIT 3"))

trips (612, 7) {'trip_id': 0, 'bike_id': 0, 'start_station_id': 0, 'end_station_id': 0, 'start_time': 0, 'duration_min': 30, 'user_type': 0}
bikes (60, 3) {'bike_id': 0, 'bike_type': 0, 'purchase_year': 0}
stations (12, 4) {'station_id': 0, 'station_name': 0, 'zone': 0, 'capacity': 1}


,bike_type,n
0,Standard,36
1,Electric,9
2,STANDARD,8
3,standard,3
4,Electric,2
5,E-bike,1
6,electric,1


,n_rows,n_trips
0,612,600


,min(duration_min),median(duration_min),max(duration_min)
0,4.0,16.0,2880.0


,start_time
0,2025-03-01 05:52:00
1,2025-03-01 06:31:00
2,2025-03-01 07:06:00


,start_time
0,01/03/2025 09:26
1,01/03/2025 18:43
2,01/03/2025 19:16


1. bike_type has 7 spellings for 2 types (case, extra space, "E-bike") → standardise to Electric / Standard
2. 12 duplicated trips (612 rows, 600 IDs) → remove exact duplicates
3. 30 trips have no duration → drop them (can't average a blank)
4. Max duration 2,880 min (48 h) → impossible ride, remove trips ≥ 24 h
5. start_time in two formats → parse both formats explicitly
6. 1 station missing capacity → ignore, not needed for this question

## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

In [9]:
bikes = tables["bikes"].copy()
bikes["bike_type"] = (bikes["bike_type"]
                      .str.strip()
                      .str.lower()
                      .replace({"e-bike": "electric"})
                      .str.title())
bikes["bike_type"].value_counts()

,count
bike_type,
Standard,47
Electric,13


In [10]:
trips = tables["trips"].copy()
before = len(trips)
trips = trips.drop_duplicates()
print("duplicates removed:", before - len(trips))

iso = pd.to_datetime(trips["start_time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
uk  = pd.to_datetime(trips["start_time"], format="%d/%m/%Y %H:%M", errors="coerce")
trips["start_time"] = iso.fillna(uk)
print("from", trips["start_time"].min(), "to", trips["start_time"].max())
print("unparsed:", trips["start_time"].isna().sum())

duplicates removed: 12
from 2025-03-01 05:52:00 to 2025-03-31 21:54:00
unparsed: 0


### What I cleaned and why

**1. Bike type: 7 spellings → 2 types**
- Found: Standard / STANDARD / standard, Electric / "Electric " (trailing space) / electric / E-bike.
- Did: removed spaces, made lowercase, mapped "e-bike" to "electric", then Title Case.
- Why: SQL treats each spelling as a different group, so the answer would have 7 bike types instead of 2.
- Check: value_counts now shows only Standard 47, Electric 13 (60 bikes in total, none lost).

**2. Duplicate trips: removed 12**
- Found: 612 rows but only 600 unique trip_ids.
- Did: drop_duplicates() (removes rows identical in every column).
- Why: a duplicated trip would be counted twice and pull the average.
- Check: 12 rows removed, 600 trips left.

**3. start_time in two formats: parsed both explicitly**
- Found: most rows are "2025-03-01 05:52:00", some are "01/03/2025 09:26" (UK day/month/year).
- Did: parsed each format separately with its exact format string, then combined them.
- Why: guessing the format can swap day and month (1 March becomes 3 January).
- Check: all dates fall between 1 and 31 March 2025; 0 unparsed.
- Note: start_time is not needed for the final answer, but I fixed it so the data is correct for any time-based follow-up.

**4. Missing durations: dropped 30 trips**
- Found: 30 trips have no duration_min.
- Did: removed them.
- Why: you cannot average a blank, and filling with the average would invent trips and hide real differences.

**5. Impossible durations: removed trips of 24 hours or more**
- Found: max duration 2,880 min (48 h); two trips of 1,440 min (24 h).
- Did: kept only trips under 1,440 minutes.
- Why: exactly 24 h and 48 h look like bikes that were never docked (system error), not real rides.
- Check: the longest trip left is 90 min, so the cut removed only those 3 records.

**6. Station capacity: 1 missing, left as is**
- Why: capacity is not used for this question, so it does not affect the answer.

**Result:** 612 rows → 567 clean trips. Registered as trips_clean and bikes_clean for SQL.

In [11]:
trips = trips.dropna(subset=["duration_min"])
trips = trips[trips["duration_min"] < 1440]
print("trips kept:", len(trips), "| longest now:", trips["duration_min"].max())

con.register("trips_clean", trips)
con.register("bikes_clean", bikes)
print("ready: trips_clean, bikes_clean")

trips kept: 567 | longest now: 90.0
ready: trips_clean, bikes_clean


### What I cleaned and why

**1. Bike type: 7 spellings → 2 types**
- Found: Standard / STANDARD / standard, Electric / "Electric " (trailing space) / electric / E-bike.
- Did: removed spaces, made lowercase, mapped "e-bike" to "electric", then Title Case.
- Why: SQL treats each spelling as a different group, so the answer would have 7 bike types instead of 2.
- Check: value_counts now shows only Standard 47, Electric 13 (60 bikes in total, none lost).

**2. Duplicate trips: removed 12**
- Found: 612 rows but only 600 unique trip_ids.
- Did: drop_duplicates() (removes rows identical in every column).
- Why: a duplicated trip would be counted twice and pull the average.
- Check: 12 rows removed, 600 trips left.

**3. start_time in two formats: parsed both explicitly**
- Found: most rows are "2025-03-01 05:52:00", some are "01/03/2025 09:26" (UK day/month/year).
- Did: parsed each format separately with its exact format string, then combined them.
- Why: guessing the format can swap day and month (1 March becomes 3 January).
- Check: all dates fall between 1 and 31 March 2025; 0 unparsed.
- Note: start_time is not needed for the final answer, but I fixed it so the data is correct for any time-based follow-up.

**4. Missing durations: dropped 30 trips**
- Found: 30 trips have no duration_min.
- Did: removed them.
- Why: you cannot average a blank, and filling with the average would invent trips and hide real differences.

**5. Impossible durations: removed trips of 24 hours or more**
- Found: max duration 2,880 min (48 h); two trips of 1,440 min (24 h).
- Did: kept only trips under 1,440 minutes.
- Why: exactly 24 h and 48 h look like bikes that were never docked (system error), not real rides.
- Check: the longest trip left is 90 min, so the cut removed only those 3 records.

**6. Station capacity: 1 missing, left as is**
- Why: capacity is not used for this question, so it does not affect the answer.

**Result:** 612 rows → 567 clean trips. Registered as trips_clean and bikes_clean for SQL.

## 3. Join and aggregate

*Join the tables and calculate the answer. Note how many rows you expect after each join, then check.*

### Join and aggregate

**Joins**
- trips_clean → bikes_clean on bike_id (each trip has one bike: many-to-one, so row count should stay 567).
- trips_clean → stations on start_station_id (the brief asks for the zone of the START station, not the end station).

**Grouping**
- GROUP BY bike_type, zone → one row per bike type × zone (2 × 5 = 10 rows).
- n_trips = COUNT(*), avg_duration_min = AVG(duration_min) rounded to 1 decimal.

**What one row represents**
All March 2025 trips that started in one zone on one bike type.

**Answer to the manager**
Electric bikes are NOT used for longer trips. They are used for SHORTER trips:
about 14–16 minutes on average vs about 19–20 minutes for standard bikes.
The pattern is the same in every zone (gap of roughly 4–6 minutes), so it does not depend much on zone.

**Caution**
Some groups are small (East electric = 11 trips), so those averages are less reliable.

In [12]:
answer = q("""
    SELECT b.bike_type,
           s.zone,
           COUNT(*)                      AS n_trips,
           ROUND(AVG(t.duration_min), 1) AS avg_duration_min
    FROM trips_clean AS t
    JOIN bikes_clean AS b ON t.bike_id = b.bike_id
    JOIN stations    AS s ON t.start_station_id = s.station_id
    GROUP BY b.bike_type, s.zone
    ORDER BY s.zone, b.bike_type
""")
answer

,bike_type,zone,n_trips,avg_duration_min
0,Electric,Centre,81,14.5
1,Standard,Centre,215,18.9
2,Electric,East,11,15.7
3,Standard,East,20,19.5
4,Electric,North,23,16.0
5,Standard,North,39,20.4
6,Electric,South,24,13.4
7,Standard,South,59,19.5
8,Electric,West,22,15.7
9,Standard,West,73,19.8


### Join and aggregate

**Joins**
- trips_clean → bikes_clean on bike_id (each trip has one bike: many-to-one, so row count should stay 567).
- trips_clean → stations on start_station_id (the brief asks for the zone of the START station, not the end station).

**Grouping**
- GROUP BY bike_type, zone → one row per bike type × zone (2 × 5 = 10 rows).
- n_trips = COUNT(*), avg_duration_min = AVG(duration_min) rounded to 1 decimal.

**What one row represents**
All March 2025 trips that started in one zone on one bike type.

**Answer to the manager**
Electric bikes are NOT used for longer trips. They are used for SHORTER trips:
about 14–16 minutes on average vs about 19–20 minutes for standard bikes.
The pattern is the same in every zone (gap of roughly 4–6 minutes), so it does not depend much on zone.

**Caution**
Some groups are small (East electric = 11 trips), so those averages are less reliable.

## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

### How I know the answer is right

1. **Count tie-out:** 567 clean trips = 567 trips summed across the 10 rows of the answer. No trip lost or double-counted.
2. **Join check:** joining trips to bikes keeps 567 rows, so every bike_id matches exactly one bike (no duplicates in bikes, no fan-out).
3. **Recalculated one cell another way:** Electric / Centre with pandas = 14.5 min from 81 trips, the same as the SQL result.
4. **Compared with no cleaning:** the raw data gives 7 "bike types" and Standard ≈ 36.9 min, inflated by the 24 h / 48 h records.
   After cleaning, Standard ≈ 19 min. The cleaning changed the conclusion, so it was necessary.

**Conclusion I trust:** electric trips are about 4–6 minutes shorter than standard trips in every zone.

In [13]:
# Check 1: every clean trip counted exactly once?
print("rows in trips_clean      :", len(trips))
print("trips counted in answer  :", answer["n_trips"].sum())

# Check 2: did the join add or lose rows?
print("rows after joining bikes :", len(trips.merge(bikes, on="bike_id")))

# Check 3: recalculate one cell a different way (pandas, not SQL)
e_centre = (trips.merge(bikes, on="bike_id")
                 .merge(tables["stations"], left_on="start_station_id", right_on="station_id")
                 .query("bike_type == 'Electric' and zone == 'Centre'")["duration_min"])
print("Electric / Centre by pandas:", round(e_centre.mean(), 1), "from", len(e_centre), "trips")

# Check 4: what would the answer be with NO cleaning?
naive = q("""SELECT b.bike_type, COUNT(*) AS n, ROUND(AVG(t.duration_min), 1) AS avg_min
             FROM trips t JOIN bikes b USING (bike_id)
             GROUP BY 1 ORDER BY n DESC""")
naive

rows in trips_clean      : 567
trips counted in answer  : 567
rows after joining bikes : 567
Electric / Centre by pandas: 14.5 from 81 trips


,bike_type,n,avg_min
0,Standard,336,36.9
1,Electric,121,14.8
2,STANDARD,71,20.5
3,Electric,34,15.3
4,standard,27,19.3
5,E-bike,12,15.5
6,electric,11,11.7


### How I know the answer is right

1. **Count tie-out:** 567 clean trips = 567 trips summed across the 10 rows of the answer. No trip lost or double-counted.
2. **Join check:** joining trips to bikes keeps 567 rows, so every bike_id matches exactly one bike (no duplicates in bikes, no fan-out).
3. **Recalculated one cell another way:** Electric / Centre with pandas = 14.5 min from 81 trips, the same as the SQL result.
4. **Compared with no cleaning:** the raw data gives 7 "bike types" and Standard ≈ 36.9 min, inflated by the 24 h / 48 h records.
   After cleaning, Standard ≈ 19 min. The cleaning changed the conclusion, so it was necessary.

**Conclusion I trust:** electric trips are about 4–6 minutes shorter than standard trips in every zone.

## 5. Prompts and AI notes

*Paste each prompt as you sent it. Note what you accepted, changed or rejected and why.*

### 5. Prompts and AI notes

**AI tool used:** Claude (Anthropic). I ran every cell myself and checked each output before moving on.

**Prompts (summarised, in order):**
1. Profile the trips, bikes and stations tables for missing values, inconsistent categories, duplicates, outliers and date formats.
2. Explain a NameError ('tables' is not defined). Cause: setup cell not run. Fixed by running setup first.
3. Explain an AttributeError ('DataFrame' has no attribute 'insa'). Cause: my typo. Fixed to isna().
4. Clean the data in short re-runnable cells: standardise bike_type, remove duplicates, parse both date formats, handle missing and extreme durations.
5. Join trips to bikes and to stations (start station) and return number of trips and average duration by bike type and zone.
6. Suggest checks to prove the final table is correct.

**What I accepted, changed or rejected, and why**
- **Accepted** the cleaning steps only after the outputs proved them:
  2 bike types (47 Standard, 13 Electric), 12 duplicates removed, all dates in March 2025 with 0 unparsed,
  567 trips kept, longest trip 90 min.
- **Rejected** the common shortcut pd.to_datetime(format="mixed", dayfirst=True), because it can read
  2025-03-01 as 3 January. I parsed the two formats explicitly instead.
- **Accepted** the SQL result only after 4 checks passed:
  567 trips in = 567 trips counted; the join kept 567 rows (no duplication);
  pandas gave the same Electric/Centre value (14.5 min, 81 trips); the uncleaned data gave a clearly wrong answer
  (7 bike types, Standard ≈ 36.9 min).
- **Changed:** no logic changes. My only edits were fixing my own run-order and typo errors.

## 6. What I could not verify

*What remains unchecked? What would you do next?*

### 6. What I could not verify

1. **The 24 h / 48 h trips (1440 and 2880 min):** I assumed they are undocked bikes (system errors), not real rides.
   I could not confirm this with the scheme. Next: ask the scheme manager how un-ended trips are recorded.
2. **The 30 missing durations:** I dropped them, but I don't know why they are missing.
   If they are missing for one bike type more than the other, dropping them could bias the averages.
   Next: check whether missing durations are spread evenly across bike types and zones.
3. **Bike type labels:** I assumed "E-bike" means electric and that the labels are correct at source.
   Next: confirm with the fleet list.
4. **Small groups:** some averages rest on few trips (East electric = 11 trips), so they are less reliable.
5. **Is the gap real or chance?** The data shows electric trips are about 4–6 min shorter, but I did not test statistical significance.
   Next: a t-test comparing electric vs standard durations (Business Statistics, Pre-Workshop 3).
6. **Only one month (March 2025):** results may differ in other seasons.
7. **User type:** user_type also has messy spellings (Member/MEMBER/member, Casual/CASUAL/"Casual "). I did not clean it because it
   is not needed for this question. If members ride e-bikes more, that could explain part of the difference.